## Clean up of data

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv('listings_detailed.csv')
display(df.head())

In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
'''
Host Characteristics:

host_is_superhost: A primary indicator of host quality.

host_response_rate / host_acceptance_rate: Measures of professional engagement.

host_identity_verified: Trust factor.

host_listings_count: Distinguishes between individual hosts and professional management companies.

host_since: Can be used to calculate "host tenure" (years of experience).


Listing Components:

room_type : Fundamental price drivers.

accommodates, bedrooms, beds, bathrooms_text: Physical capacity of the unit.

amenities: You can count the number of amenities or look for specific high-value ones (e.g., "Wifi," "Pool").

neighbourhood_cleansed: Essential for comparing across cities.


Review & Quality Metrics:

number_of_reviews: Popularity and social proof.

instant_bookable: Ease of booking.


Target Variable:

price: This is what you are predicting or estimating.

'''

In [ ]:
cols_to_keep = [
    'host_is_superhost',
    'host_response_rate',
    'host_acceptance_rate',
    'host_identity_verified',
    'host_listings_count',
    'host_since',
    'room_type',
    'accommodates',
    'bedrooms',
    'beds',
    'bathrooms_text',
    'amenities',
    'neighbourhood_cleansed',
    'number_of_reviews',
    'instant_bookable',
    'price'
]

In [ ]:
la_df = df[cols_to_keep].copy()
la_df.head()

In [ ]:
la_df.isna().sum()

In [ ]:
#host_is_superhost: * Action: Usually, if this is null, the host is not a Superhost. You can safely fill these with "f" (false).
la_df['host_is_superhost'] = la_df['host_is_superhost'].fillna('f')

In [ ]:
'''
host_response_rate & host_acceptance_rate:
-->
second "flag" column (e.g., has_response_rate: 0 or 1) to see
if not having a rate impacts price.

why: because these rates can give severe advantage to someone with 95% rate over
someone with 85% rate.

'''

la_df['has_response_rate'] = la_df['host_response_rate'].notna().astype(int)
la_df['has_acceptance_rate'] = la_df['host_acceptance_rate'].notna().astype(int)


In [ ]:
la_df.drop(columns=['host_response_rate', 'host_acceptance_rate'], inplace=True)

In [ ]:
la_df.isna().sum()

In [ ]:
la_df.head()

In [ ]:
'''
Action: Use the accommodates value to estimate.
If accommodates is 1 or 2 and bedrooms is null, it's safe to impute a 1.
'''
import numpy as np

# Logic for Bedrooms: All Nulls become 1 (treating Studios as 1-bed units)
la_df['bedrooms'] = la_df['bedrooms'].fillna(1)

# Logic for Beds: If Null, use half of 'accommodates' (rounded up)
# Example: Accommodates 1 or 2 -> 1 bed. Accommodates 3 or 4 -> 2 beds.
la_df['beds'] = la_df['beds'].fillna((la_df['accommodates'] / 2).apply(np.ceil))


In [ ]:
la_df.isna().sum()

In [ ]:
# now just remove all rows with nulls

la_df.dropna(inplace=True)

In [ ]:
la_df.isna().sum()

In [ ]:
la_df.head()

In [ ]:
#for amenities --> just count the number of amenities and make a column named --> amenities_count

la_df['amenities_count'] = la_df['amenities'].str.count(',') + 1
la_df.drop(columns=['amenities'], inplace=True)

In [ ]:
la_df.head()

In [ ]:
la_df.head()

Now we begin fixing the data (cleaning)

In [ ]:
#converting host_is_superhost, host_identity_verified, instant_bookable
la_df['host_is_superhost'] = la_df['host_is_superhost'].map({'t': 1, '  f': 0  ,'f': 0})
la_df['instant_bookable'] = la_df['instant_bookable'].map({'t': 1, 'f': 0})

In [ ]:
#look for all uniques in host_identity verified
la_df['host_identity_verified'].unique()
#now convert
la_df['host_identity_verified'] = la_df['host_identity_verified'].map({'t': 1, 'f': 0})


In [ ]:
la_df.head()

In [ ]:
#adjust the host_since --> for actual numerical value --> in years
import pandas as pd
from datetime import datetime

la_df['host_since'] = la_df['host_since'].astype(str).str[:10]

la_df['host_since'] = pd.to_datetime(la_df['host_since'], errors='coerce')

reference_date = pd.to_datetime('2026-04-11')
la_df['host_since'] = (reference_date - la_df['host_since']).dt.days / 365.25



In [ ]:
la_df.head()

In [ ]:
#chnage the room_type to be numerical --> (One-Hot Encoding) to avoid traps with 0 and 5 being numerically different
# nyc_df['room_type'].unique()

# This creates the room_type_Private room, etc. as 1s and 0s
la_df = pd.get_dummies(la_df, columns=['room_type'], dtype=int)



In [ ]:
la_df.head()

In [ ]:
la_df.isna().sum()

In [ ]:
#fix the bathrooms text

# 1. Create 'bathrooms' column by grabbing the first number found in the text
# This handles "3.5 baths", "1 bath", "11.5 shared baths", etc.
la_df['bathrooms'] = la_df['bathrooms_text'].str.extract('(\d+\.?\d*)').astype(float)

# 2. Fix the "Half-bath" cases (since they have no number, extract makes them NaN)
# If the text says "half", we just set the number to 0.5
la_df.loc[la_df['bathrooms_text'].str.contains('half', case=False, na=False), 'bathrooms'] = 0.5

# 3. Create 'is_shared_bath' (1 if it's shared, 0 if it's private)
# This is a simple "True/False" check converted to "1/0"
la_df['is_shared_bath'] = la_df['bathrooms_text'].str.contains('shared', case=False, na=False).astype(int)

# 4. Fill any remaining blanks with 1 (the most common bathroom count)
la_df['bathrooms'] = la_df['bathrooms'].fillna(1)

In [ ]:
la_df.drop(columns=['bathrooms_text'], inplace=True)

In [ ]:
la_df.isna().sum()

In [ ]:
la_df.head()

In [ ]:
#fix the neighbourhood_cleansed
la_df['neighbourhood_cleansed'].unique()

In [ ]:
neighborhood_groups = {
    'Coastal': [
        'Malibu', 'Santa Monica', 'Venice', 'Pacific Palisades', 'Manhattan Beach',
        'Hermosa Beach', 'Marina del Rey', 'Playa del Rey', 'Avalon', 'Rancho Palos Verdes',
        'San Pedro', 'Palos Verdes Estates', 'Rolling Hills Estates', 'Rolling Hills',
        'El Segundo', 'Redondo Beach', 'Del Aire', 'Unincorporated Catalina Island'
    ],
    'Westside_Hollywood': [
        'Beverly Hills', 'Bel-Air', 'West Hollywood', 'Hollywood Hills', 'Hollywood Hills West',
        'Brentwood', 'Westwood', 'Culver City', 'Beverly Grove', 'Mar Vista', 'Sawtelle',
        'Cheviot Hills', 'Beverly Crest', 'Pico-Robertson', 'Pacific Palisades', 'Century City',
        'Rancho Park', 'Cheviot Hills', 'Windsor Square', 'Hancock Park', 'Beverlywood',
        'Bel-Air', 'Universal City', 'West Los Angeles', 'Del Rey', 'Playa Vista'
    ],
    'Central_LA': [
        'Downtown', 'Koreatown', 'Hollywood', 'Silver Lake', 'Echo Park', 'Mid-Wilshire',
        'Westlake', 'Los Feliz', 'Chinatown', 'Boyle Heights', 'East Hollywood', 'Carthay',
        'Pico-Union', 'Harvard Heights', 'Larchmont', 'Arlington Heights', 'University Park',
        'West Adams', 'Jefferson Park', 'Exposition Park', 'Adams-Normandie', 'Elysian Valley',
        'Highland Park', 'Eagle Rock', 'Atwater Village', 'Glassell Park', 'Mount Washington',
        'Cypress Park', 'Elysian Park', 'Montecito Heights', 'Lincoln Heights', 'Historic South-Central'
    ],
    'The_Valley': [
        'Sherman Oaks', 'Encino', 'Northridge', 'Woodland Hills', 'Van Nuys', 'Burbank',
        'Studio City', 'Reseda', 'Canoga Park', 'North Hollywood', 'Valley Village',
        'Valley Glen', 'Van Nuys', 'Tarzana', 'Chatsworth', 'North Hills', 'Granada Hills',
        'Porter Ranch', 'San Fernando', 'Pacoima', 'Sylmar', 'Mission Hills', 'Arleta',
        'Panorama City', 'Sun Valley', 'Sunland', 'Tujunga', 'Shadow Hills', 'Lake Balboa',
        'Winnetka', 'West Hills', 'Toluca Lake', 'Porter Ranch', 'Lake View Terrace',
        'Sepulveda Basin', 'Tujunga Canyons'
    ],
    'South_LA_Harbor': [
        'Long Beach', 'Torrance', 'San Pedro', 'Carson', 'Gardena', 'Inglewood',
        'Hawthorne', 'Compton', 'Watts', 'Lennox', 'Lawndale', 'Lomita', 'Harbor City',
        'Wilmington', 'Harbor Gateway', 'West Carson', 'Athens', 'Westmont', 'Florence',
        'Florence-Firestone', 'Green Meadows', 'South Park', 'Central-Alameda', 'Vermont Square',
        'Vermont Vista', 'Vermont Knolls', 'Vermont-Slauson', 'Hyde Park', 'Leimert Park',
        'View Park-Windsor Hills', 'Baldwin Hills/Crenshaw', 'Ladera Heights', 'Gramercy Park',
        'Manchester Square', 'Chesterfield Square', 'Harvard Park', 'Walnut Park', 'Lynwood',
        'South Gate', 'Cudahy', 'Bell', 'Bell Gardens', 'Huntington Park', 'Vernon', 'Commerce',
        'Signal Hill', 'Lakewood', 'Bellflower', 'Norwalk', 'Paramount', 'Cerritos', 'Artesia',
        'Hawaiian Gardens', 'Alondra Park', 'Willowbrook', 'Rancho Dominguez'
    ],
    'East_LA_SGV': [
        'Pasadena', 'Alhambra', 'Monterey Park', 'East Los Angeles', 'El Monte',
        'West Covina', 'Diamond Bar', 'Glendale', 'South Pasadena', 'San Marino',
        'San Gabriel', 'Temple City', 'Arcadia', 'Sierra Madre', 'Monrovia', 'Duarte',
        'Azusa', 'Glendora', 'Claremont', 'La Verne', 'San Dimas', 'Pomona', 'Walnut',
        'Rowland Heights', 'Hacienda Heights', 'La Puente', 'Valinda', 'Bassett',
        'West Puente Valley', 'Avocado Heights', 'South El Monte', 'Rosemead', 'Pico Rivera',
        'Whittier', 'Montebello', 'South San Gabriel', 'East San Gabriel', 'Mayflower Village',
        'Irwindale', 'Baldwin Park', 'Citrus', 'Charter Oak', 'South San Jose Hills',
        'South Whittier', 'East Whittier', 'North Whittier', 'La Mirada', 'Santa Fe Springs',
        'Downey', 'La Habra Heights', 'Walnut Park', 'East Pasadena', 'South Whittier',
        'Altadena', 'La Crescenta-Montrose', 'La Canada Flintridge'
    ],
    'North_County': [
        'Santa Clarita', 'Lancaster', 'Palmdale', 'Castaic', 'Acton', 'Agua Dulce',
        'Stevenson Ranch', 'Val Verde', 'Castaic Canyons', 'Hasley Canyon', 'Quartz Hill',
        'Lake Los Angeles', 'Sun Village', 'Littlerock', 'Elizabeth Lake', 'Lake Hughes',
        'Leona Valley', 'Green Valley', 'Desert View Highlands', 'Northwest Antelope Valley',
        'Northeast Antelope Valley', 'Southeast Antelope Valley', 'Northwest Palmdale',
        'Agoura Hills', 'Calabasas', 'Hidden Hills', 'Westlake Village', 'Topanga',
        'Unincorporated Santa Monica Mountains', 'Unincorporated Santa Susana Mountains',
        'Ridge Route', 'Angeles Crest', 'Lopez/Kagel Canyons', 'Vincent'
    ]
}

In [ ]:
# Create the flat mapping dictionary
flat_map = {nb: region for region, nbs in neighborhood_groups.items() for nb in nbs}

# Map the column
la_df['region'] = la_df['neighbourhood_cleansed'].map(flat_map).fillna('Other_Residential')

# Final One-Hot Encoding
la_df = pd.get_dummies(la_df, columns=['region'], drop_first=True, dtype=int)

In [ ]:
la_df.drop(columns=['neighbourhood_cleansed'], inplace=True)

In [ ]:
la_df.head()

In [ ]:
la_df.columns

In [ ]:
#now begin scaling

from sklearn.preprocessing import StandardScaler

# 1. Clean price first (ensure it's a float, no $ or ,)
la_df['price'] = la_df['price'].replace('[\$,]', '', regex=True).astype(float)

# 2. Define the numerical columns that need scaling
cols_to_scale = [
    'host_listings_count',
    'host_since',
    'accommodates',
    'bedrooms',
    'beds',
    'number_of_reviews',
    'bathrooms'
]

# 3. Initialize and apply the scaler
scaler = StandardScaler()
la_df[cols_to_scale] = scaler.fit_transform(la_df[cols_to_scale])



In [ ]:
la_df.head()

In [ ]:
la_df.columns

In [ ]:
la_df.isna().sum()

## Model Trainin

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_squared_error, r2_score, classification_report
import warnings
warnings.filterwarnings('ignore')



In [ ]:
la_df.head()

In [ ]:
# Target column check
y = la_df['price']

# features syncing
FEATURES = [
    'host_is_superhost', 'host_identity_verified', 'host_listings_count',
    'host_since', 'accommodates', 'bedrooms', 'beds', 'number_of_reviews',
    'instant_bookable', 'bathrooms', 'is_shared_bath',
    'room_type_Private room', 'room_type_Shared room', 'room_type_Hotel room',
    'region_Coastal', 'region_East_LA_SGV', 'region_North_County',
    'region_Other_Residential', 'region_South_LA_Harbor', 'region_The_Valley',
    'region_Westside_Hollywood'
]

# handling any possible NaN values thought i doubt there are any left!
X = la_df[FEATURES].fillna(0)

# 4. The Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

linear regression:

To get the Pr, Re, and F1 points, you need to slightly tweak your goals. Instead of asking "How much does this feature change the price?", you ask "Which features help us predict if a house is Luxury vs. Standard



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Create the 'Class' (e.g., Top 25% of houses are 'Luxury')
threshold = np.percentile(y_train, 75)
y_train_class = (y_train > threshold).astype(int)
y_test_class = (y_test > threshold).astype(int)

# 2. Use Logistic Regression (This is for the 'Prediction Task')
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train_class)

# 3. Predict and get the Pr/F1 metrics your prof wants
y_pred_class = log_reg.predict(X_test)
print(classification_report(y_test_class, y_pred_class))

# 4. Feature Impact (Same as before, but for classification)
log_coef = pd.Series(log_reg.coef_[0], index=FEATURES).sort_values(ascending=True)
log_coef.plot(kind='barh', color='darkmagenta')
plt.title('Features that Predict "Luxury" Status')
plt.show()

precision: of all modeled/predicted [x] how many were acutally [x]

recall : of all the actual [x]'s , how many did the model find?

f1: overall grade for precision and recall

Overall model accuracy: 0.84
Overall Precision: 0.82
Overall recall: 0.79

              precision    recall  f1-score   support

           0       0.89      0.95      0.92      6355
           1       0.80      0.65      0.71      2127

    accuracy                           0.87      8482
   macro avg       0.84      0.80      0.81      8482
weighted avg       0.87      0.87      0.87      8482

decision trees:


1. The Power of Space (Bedrooms/Bathrooms)
Decision Tree: bedrooms is by far the most important feature (nearly 0.6 importance score). This means the tree uses the number of bedrooms as its first and most important "split" to decide price.

Linear Regression: bedrooms and bathrooms both have strong positive coefficients. This confirms that in LA, physical capacity is the primary driver of value.

2. Location, Location, Location
Coastal & Westside: Both models agree that region_Coastal and region_Westside_Hollywood are huge value boosters. In your regression, region_Coastal is the strongest positive geographic coefficient.

Inland Regions: Notice how region_East_LA_SGV and region_South_LA_Harbor have negative or near-zero coefficients. This highlights the massive price gap between the coast and the inland valleys in LA.

3. The "Discount" Features
Shared Spaces: Look at room_type_Shared room and is_shared_bath. They have massive negative coefficients in your regression. This is a crucial finding: even if a house is in Malibu, if the bathroom is shared, the "penalty" to the price is significant.

Hotel Rooms: Interestingly, room_type_Hotel room has a high positive coefficient. This suggests that boutique hotel listings on Airbnb are commanding a premium over residential homes.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Initialize and Fit the Classifier
# We use y_train_class (the 0/1 labels you created from the 75th percentile)
dt_clf = DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=42)
dt_clf.fit(X_train, y_train_class)

# 2. Make Class Predictions
y_pred_dt_class = dt_clf.predict(X_test)

# 3. THE EVALUATION TABLE (This is the F1, Pr, Re table!)
print("=== DECISION TREE CLASSIFIER (Prediction Task) ===")
print(classification_report(y_test_class, y_pred_dt_class, target_names=['Budget', 'Luxury']))

# 4. Optional: Visual Confusion Matrix (Shows exactly where the model guessed wrong)
cm = confusion_matrix(y_test_class, y_pred_dt_class)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Budget', 'Luxury'])
disp.plot(cmap='Oranges')
plt.title('Decision Tree Confusion Matrix')
plt.show()

# 5. Feature Importance
dt_imp = pd.Series(dt_clf.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
dt_imp.head(15).sort_values().plot(kind='barh', color='darkorange')
plt.title('Decision Tree — Top 15 Features Predicting "Luxury"')
plt.show()

# 6. Visualizing the Tree
plt.figure(figsize=(20,10))
plot_tree(dt_clf, max_depth=2, feature_names=FEATURES,
          class_names=['Budget', 'Luxury'], filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree Structure (Top 2 Levels)")
plt.show()

precision: of all modeled/predicted [x] how many were acutally [x]

recall : of all the actual [x]'s , how many did the model find?

f1: overall grade for precision and recall

=== DECISION TREE CLASSIFIER (Prediction Task) ===
              precision    recall  f1-score   support

      Budget       0.90      0.94      0.92      6355
      Luxury       0.78      0.68      0.72      2127

    accuracy                           0.87      8482
   macro avg       0.84      0.81      0.82      8482
weighted avg       0.87      0.87      0.87      8482

random forest

1. The "Bath Premium" vs. The "Bedroom Logic"
In many cities, bedrooms are everything. But in LA, look how high bathrooms and is_shared_bath are.

The Finding: LA has a massive market for "Luxury" and "Entire Home" privacy. The "Importance" of bathrooms suggests that in LA, having your own private, high-end bathroom is a bigger deal for price than in denser cities where people might tolerate shared facilities more.

2. The Geographic "Cliffs"
Notice the gap between region_Coastal / region_Westside_Hollywood and everything else.

The Finding: Location in LA isn't a "sliding scale"; it's a cliff. The model finds it very important to distinguish if a property is on the "right" side of the 405 freeway. If you look at region_North_County or region_East_LA_SGV, they are almost at the bottom of the importance list.

Research Interpretation: In LA, being in a "prestigious" zone (Coastal/Westside) is a primary data split, whereas the specific differences between the various inland valleys don't actually help the model predict price that much.

3. "Host Scale" over "Host Reputation"
Look at host_listings_count vs. host_is_superhost and number_of_reviews.

The Finding: host_listings_count (professional management) is more important for predicting price than host_is_superhost or how many reviews someone has.

Research Interpretation: The LA Airbnb market is heavily professionalized. High-priced listings are driven by "Power Hosts" with multiple properties, not necessarily by the "mom-and-pop" superhosts with one highly-reviewed spare room.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Initialize and Fit the Classifier
# We use y_train_class (the 0/1 labels)
# n_estimators=200 means we are building 200 different trees to vote on the result
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train_class)

# 2. Make Class Predictions
y_pred_rf_class = rf_clf.predict(X_test)

# 3. THE EVALUATION TABLE (Pr, Re, F1)
print("=== RANDOM FOREST CLASSIFIER (Prediction Task) ===")
print(classification_report(y_test_class, y_pred_rf_class, target_names=['Budget', 'Luxury']))

# 4. Visual Confusion Matrix
cm_rf = confusion_matrix(y_test_class, y_pred_rf_class)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=['Budget', 'Luxury'])
disp_rf.plot(cmap='Greens')
plt.title('Random Forest Confusion Matrix')
plt.show()

# 5. Feature Importance
rf_imp = pd.Series(rf_clf.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
rf_imp.head(15).sort_values().plot(kind='barh', color='seagreen')
plt.title('Random Forest — Top 15 Features Predicting "Luxury"')
plt.xlabel('Importance Score')
plt.show()

print("\nTop 10 features driving Luxury Classification (Random Forest):")
print(rf_imp.head(10))

precision: of all modeled/predicted [x] how many were acutally [x]

recall : of all the actual [x]'s , how many did the model find?

f1: overall grade for precision and recall

=== RANDOM FOREST CLASSIFIER (Prediction Task) ===
              precision    recall  f1-score   support

      Budget       0.89      0.95      0.92      6355
      Luxury       0.80      0.67      0.73      2127

    accuracy                           0.88      8482
   macro avg       0.85      0.81      0.82      8482
weighted avg       0.87      0.88      0.87      8482

kmeans --> cluster model

In Clustering, the question becomes: "What are the natural 'types' of Airbnbs in Los Angeles?" Instead of forcing the data into "Luxury" or "Budget," the model might discover:

Clustering revealed that the LA market is structurally divided into three segments: Large Estates, Budget Shared Spaces, and Professional Management groups. This proves that while physical size dictates price, host professionalization is a key hidden factor in the LA Airbnb ecosystem."


####Cluster 0: The "Full-Sized Estates" (The Luxury Group)
Look at the features that are positive and high:

- Accommodates (1.51),
- Bedrooms (1.58),
- Beds (1.45), and
- Bathrooms (1.24).

These are significantly higher than the other groups.

Conclusion: This cluster represents large, multi-bedroom homes or entire houses. Since your Purity was 0.84, this cluster almost certainly contains most of your "Luxury" category.


#### Cluster 1: The "Economy/Shared Stays" (The Budget Group)
Look at where the numbers are negative or small:

- Accommodates (-0.39) and
- Bedrooms (-0.41) are negative (meaning below average for your dataset).
- is_shared_bath (0.16) is the highest here.
- room_type_Private room (0.35) is also the highest here.

Conclusion: This group represents smaller stays—private rooms in houses or studios. These are your "Budget" listings.



####Cluster 2: The "High-Volume Professionals" (The Business Group)
This is a fascinating "discovery" the model made:

- host_listings_count (6.86): This is massive compared to the others!
- host_identity_verified (1.00): Every single one is verified.
- accommodates (-0.46): They are small units.

Conclusion: This cluster isn't defined by the house size, but by the Host. These are likely professional property management companies or "ghost hotels" that list dozens of small units across Hollywood and the Westside.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# Initialize and Fit K-Means
# We'll try 3 clusters (perhaps Budget, Mid-tier, and Luxury)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X) # Notice we only use X! No 'y' here.

# Evaluation: SSE (Inertia)
# This is how 'tight' the clusters are. Lower is generally better.
sse = kmeans.inertia_
print(f"Cluster SSE (Inertia): {sse:.2f}")

# Evaluation: Silhouette Score
# Measures how well-separated the clusters are (-1 to 1).
sil_score = silhouette_score(X, clusters)
print(f"Silhouette Score: {sil_score:.4f}")

# Evaluation: Purity Function
# Since sklearn doesn't have 'purity' built-in, we calculate it manually
def calculate_purity(y_true, cluster_labels):
    matrix = confusion_matrix(y_true, cluster_labels)
    return np.sum(np.amax(matrix, axis=0)) / np.sum(matrix)

# We compare the clusters to the 'Luxury' classes we made for the Prediction Task
purity = calculate_purity(y_test_class, kmeans.predict(X_test))
print(f"Cluster Purity (relative to Luxury class): {purity:.4f}")

In [ ]:
# Look at the "average" house in each cluster
cluster_summary = pd.DataFrame(kmeans.cluster_centers_, columns=FEATURES)

# Transpose it so it's easier to read
print("=== CHARACTERISTICS OF EACH CLUSTER ===")
print(cluster_summary.T)

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Reduce dimensions to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X) # Use your scaled features X

# 2. Create the Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=clusters, palette='viridis', s=70, alpha=0.8)

plt.title('Airbnb Market Segments (PCA Visualization)', fontsize=15)
plt.xlabel('Principal Component 1 (Size/Price Factor)')
plt.ylabel('Principal Component 2 (Location/Type Factor)')
plt.legend(title='Cluster ID')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('airbnb_clusters_pca.png')

now kmeans + tsne

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# 1. Outlier Removal (using the original la_df)
Q1 = la_df['price'].quantile(0.25)
Q3 = la_df['price'].quantile(0.75)
IQR = Q3 - Q1
la_df_clean = la_df[(la_df['price'] >= Q1 - 1.5*IQR) & (la_df['price'] <= Q3 + 1.5*IQR)].copy()

# 2. Define Feature Groups
# We need to make sure we don't try to log-transform a column that isn't there
num_cols = ['host_listings_count', 'host_since', 'accommodates', 'bedrooms', 'beds', 'number_of_reviews']
bin_cols = [
    'host_is_superhost', 'host_identity_verified', 'instant_bookable', 'is_shared_bath',
    'room_type_Private room', 'room_type_Shared room', 'room_type_Hotel room',
    'region_Coastal', 'region_East_LA_SGV', 'region_North_County',
    'region_Other_Residential', 'region_South_LA_Harbor', 'region_The_Valley',
    'region_Westside_Hollywood'
]

# Create X_clean using whatever is available
X_clean = la_df_clean[num_cols + bin_cols].fillna(0).copy()

# Add and Transform Price specifically
X_clean['price_log'] = np.log1p(la_df_clean['price'])
num_cols_with_price = num_cols + ['price_log']

# 3. Scaling: Only on continuous numbers
scaler = StandardScaler()
X_scaled_num = scaler.fit_transform(X_clean[num_cols_with_price])

# Combine scaled numbers with the 0/1 binary columns
X_final = np.hstack([X_scaled_num, X_clean[bin_cols].values])

# 4. K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_final)

print(f"Cleaned Silhouette Score: {silhouette_score(X_final, clusters):.4f}")



In [ ]:
# 5. Visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# PCA
pca_coords = PCA(n_components=2).fit_transform(X_final)
sns.scatterplot(x=pca_coords[:,0], y=pca_coords[:,1], hue=clusters, palette='viridis', ax=ax1, alpha=0.5)
ax1.set_title('PCA: Market Segments (After Outlier Removal)')

# t-SNE (Running on a 5k sample for speed)
sample_size = min(5000, len(X_final))
sample_idx = np.random.choice(len(X_final), sample_size, replace=False)
tsne_coords = TSNE(n_components=2, perplexity=40, random_state=42).fit_transform(X_final[sample_idx])

sns.scatterplot(x=tsne_coords[:,0], y=tsne_coords[:,1], hue=clusters[sample_idx], palette='viridis', ax=ax2, alpha=0.6)
ax2.set_title('t-SNE: Local Density Clusters')

plt.show()